
# Baseline – Regelbaserad modell

## Syfte
Vi använder en enkel regelbaserad baseline som jämförelse mot våra maskininlärningsmodeller. Syftet är att undersöka om modellerna kan identifiera fler avbokningar än en enkel regel.

## Metod
Baselinen använder `lead_time`, alltså antalet dagar mellan bokning och planerad ankomst, som riskmått.

Vi antar att bokningar med längre `lead_time` har högre risk att avbokas.

1. Sortera bokningarna efter `lead_time`, från högst till lägst.
2. Flagga de 20 % av bokningarna som har längst `lead_time`.
3. Jämför de flaggade bokningarna med de faktiska avbokningarna.

Baselinen tränas inte och använder inte information om faktiska avbokningar när den väljer vilka bokningar som ska flaggas.

Vi använder samma valideringsdata och samma andel flaggade bokningar som vid jämförelsen med ML-modellerna.


In [1]:
import os

# Byt till projektroten om notebooken körs från notebooks-mappen.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

c:\Inlämningar\data-science-group-project


In [2]:
from src.data_split import load_cleaned_data, get_splits
from src.evaluation import flag_top_share, evaluate_top_share

# Läs in städad data och hämta valideringsdatan.
df = load_cleaned_data()
val = get_splits(df)["val"]

print(f"Antal bokningar i valideringsdatan: {len(val)}")

2026-10-15 10:01:12 [INFO] src.data_split: Läser in städad data från data\processed\hotel_bookings_cleaned.csv
2026-10-15 10:01:13 [INFO] src.data_split: Split vid 2016-07-01: 40414 träning (avbokade 0.308), 11643 utvärdering (avbokade 0.353)
2026-10-15 10:01:13 [INFO] src.data_split: Split vid 2016-10-01: 55393 träning (avbokade 0.321), 14879 utvärdering (avbokade 0.396)
Antal bokningar i valideringsdatan: 11643


In [3]:
# Flagga de 20 % av bokningarna som har längst lead_time.
predictions = flag_top_share(val["lead_time"], share=0.20)

# Visa hur många bokningar som flaggades.
print(f"Antal bokningar: {len(val)}")
print(f"Antal flaggade: {predictions.sum()}")
print(f"Andel flaggade: {predictions.mean():.1%}")

Antal bokningar: 11643
Antal flaggade: 2329
Andel flaggade: 20.0%


In [4]:
# Utvärdera baselinen mot de faktiska avbokningarna.
results = evaluate_top_share(
    y_true=val["is_canceled"],
    scores=val["lead_time"],
    share=0.20
)

# Visa resultaten i procent.
print(f"Precision (20 % flaggade): {results['precision']:.2%}")
print(f"Recall (20 % flaggade): {results['recall']:.2%}")
print(f"F1-score (20 % flaggade): {results['f1_score']:.2%}")
print(f"Korrekt flaggade avbokningar: {results['korrekt_flaggade']}")

Precision (20 % flaggade): 54.36%
Recall (20 % flaggade): 30.83%
F1-score (20 % flaggade): 39.34%
Korrekt flaggade avbokningar: 1266



## Resultat och slutsats

Baselinen flaggade de 20 % av bokningarna som hade längst `lead_time`.

Av totalt 11 643 bokningar flaggades 2 329. Av dessa blev 1 266 faktiskt avbokade.

**Precision (54,36 %):** Ungefär 54 av 100 flaggade bokningar blev avbokade.

**Recall (30,83 %):** Genom att flagga 20 % av bokningarna identifierade vi ungefär 31 % av alla faktiska avbokningar.

**F1-score (39,34 %):** Ett sammanvägt mått av precision och recall som visar balansen mellan hur många av de flaggade bokningarna som faktiskt avbokades och hur stor andel av alla avbokningar vi lyckades identifiera.

### Begränsningar

Baselinen använder endast `lead_time` och tar inte hänsyn till andra faktorer som kan påverka avbokningsrisken.

Den bygger på ett enkelt antagande om att längre tid mellan bokning och ankomst innebär högre avbokningsrisk.

Resultatet används som jämförelsevärde för att undersöka om våra ML-modeller kan identifiera fler avbokningar när samma andel bokningar flaggas.
